# CircuitVision · Round 3 Expanded Laboratory Components (Physical Objects)

**Uses the six-source dataset audit from 2026-10-19. This notebook is for a new experimental Round 3 model. It does not alter the live Round 2 website.**

- Retrains using **4 physically labeled datasets**: the two original Round 2 sources + expanded electronics parts + laboratory instruments.
- Preserves the names and numeric indices of **all 57 existing Round 2 classes**, appending new categories after them.
- Explicitly reconciles overlapping class names, e.g. 9V batteries, Arduino boards and breadboards. It **does not** pretend to identify resistor values from object boxes.
- Uses a T4 GPU, held-out tests and an export checkpoint before any website update.
- **Does not** add unrepresented 7400/7408/7432/7486 IC part numbers. Specific part identification usually needs readable chip markings/OCR.
- **Does not** mix the two separate *resistor band color* datasets into this physical-object model.

Your audit report counted 9,382 EESOB images, 18,556 original electronics images, 5,077 expanded electronics images, and 1,322 instrument images **including augmentation and non-training splits**. Actual diversity may be lower. Free Colab uses limits per source to stay manageable.

**Important:** The previously reported Round 2 test mAP@50 (49.87%) and Round 3 scores are NOT directly comparable unless evaluated on the same exact frozen holdout with a harmonized class mapping.

## 1 — Install dependencies and confirm a GPU

Colab: **Runtime → Change runtime type → T4 GPU**. Then **Runtime → Run all**. The notebook requires a real CUDA GPU; it will stop before training otherwise.

In [ ]:
%pip -q install "ultralytics>=8.3,<9" "roboflow>=1.1" "PyYAML>=6" "onnx>=1.15" "onnxruntime>=1.18" "onnxslim>=0.1"
import torch, pathlib, json, os, gc, hashlib, io, zipfile, shutil, random, re, math
from pathlib import Path
from getpass import getpass
from collections import Counter
import yaml
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'MISSING')
assert torch.cuda.is_available(),'Please enable T4 GPU under Runtime settings before training.'

## 2 — Get the audited physical-component datasets

If you stayed in the same Colab runtime as the audit, existing files under /content/circuitvision_dataset_audit are reused. Otherwise, enter your Roboflow key privately. Download failures stop training—no silent data gaps.

In [ ]:
from roboflow import Roboflow
SOURCES=[
 dict(name='eesob_original',workspace='android-yolo',project='eesob',version=4,required=True),
 dict(name='electronics_original',workspace='vishnupriyas-workspace',project='electronics-components-sveck',version=7,required=True),
 dict(name='electronic_component_detection',workspace='personal-8hq68',project='electronic-component-detection-m6sop',version=3,required=True),
 dict(name='lab_instruments',workspace='myspace-natcf',project='deteksi-alat-elektronik',version=2,required=True),
]
DATA_ROOT=Path('/content/circuitvision_dataset_audit')
DATA_ROOT.mkdir(parents=True,exist_ok=True)
roots={}
rf=None
for source in SOURCES:
    folder=DATA_ROOT/source['name']
    existing=next(iter(folder.rglob('data.yaml')),None) if folder.exists() else None
    if existing is not None:
        roots[source['name']]=existing.parent
        print('Reused:',source['name'])
        continue
    if rf is None:
        api_key=getpass('Roboflow API key (hidden): ').strip()
        assert api_key,'Roboflow API key required'
        rf=Roboflow(api_key=api_key)
        del api_key
    problems=[]
    for fmt in ('yolov11','yolov8'):
        try:
            record=rf.workspace(source['workspace']).project(source['project']).version(source['version']).download(fmt,location=str(folder))
            roots[source['name']]=Path(record.location)
            print('Downloaded:',source['name'],fmt)
            break
        except Exception as exc:
            problems.append(fmt+': '+str(exc)[:250])
    if source['name'] not in roots:
        raise RuntimeError('Cannot download '+source['name']+': '+'; '.join(problems))
del rf
assert len(roots)==4
print('All four physical-component sources available')

## 3 — Load and verify your **real Round 2** checkpoint

Downloads the exact currently deployed Round 2 training ZIP from your GitHub repository and checks its pinned SHA256. The class list in the PyTorch checkpoint must agree with classes.json.

In [ ]:
from urllib.request import urlopen,Request
from ultralytics import YOLO
MODEL_URL='https://raw.githubusercontent.com/aquinogeraldedmiranda-art/CircuitVision-YOLO/main/models/CircuitVision_Accuracy_Round2.zip'
PINNED_SHA256='40b97bd83288d8dcc15082019ff335612683cd09d90997ca55ff27ca5d03cae3'
WEIGHTS_DIR=Path('/content/circuitvision_round2_verified')
WEIGHTS_DIR.mkdir(exist_ok=True)
MODEL_PT=WEIGHTS_DIR/'best.pt'
if not MODEL_PT.exists():
    raw=urlopen(Request(MODEL_URL,headers={'User-Agent':'CircuitVision-Round3-Colab'}),timeout=180).read(25_000_000)
    digest=hashlib.sha256(raw).hexdigest()
    assert digest==PINNED_SHA256,f'Wrong Round 2 archive SHA: {digest}'
    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        assert 'best.pt' in z.namelist() and 'classes.json' in z.namelist()
        MODEL_PT.write_bytes(z.read('best.pt'))
        (WEIGHTS_DIR/'classes.json').write_bytes(z.read('classes.json'))
    del raw
baseline=YOLO(str(MODEL_PT))
old_names=[str(baseline.names[i]) for i in range(len(baseline.names))]
assert len(old_names)==57,'Round 2 model must contain 57 prior categories'
original_map=json.loads((WEIGHTS_DIR/'classes.json').read_text())
assert [original_map[str(i)] for i in range(57)]==old_names,'Checkpoint class mapping differs from saved classes.json'
print('Verified original 57 classes, preserving exact index order')
print(old_names)

## 4 — Define an explicit, defensible class-name mapping

Parts that cannot be distinguished visually (e.g. capacitor printed values) are merged into their broad category rather than given a false numeric value. New instrument categories are added. Every source class must be mapped; unexpected spelling stops the merge.

In [ ]:
def norm(s):
    return re.sub(r'\s+',' ',re.sub(r'[_-]+',' ',str(s).strip().lower())).strip()
existing={norm(n):n for n in old_names}
assert len(existing)==57
EXTRA_ALIASES={
 '1-5-Volt-Battery':'Battery',
 '3-3-Volt-Battery':'Battery',
 '7-Segment-Display':'7-segment',
 '9-Volt-Battery':'Battery 9V',
 'Arduino-Mega':'Arduino Mega',
 'Arduino-Nano':'Arduino Nano',
 'Arduino-Uno':'Arduino Uno',
 'BJT-Transistor':'Transistor',
 'Bluetooth-Module':'Bluetooth',
 'Breadboard':'Breadboard',
 'Bridge-Rectifier':'Bridge Rectifier',
 'Buck-Converter':'Power Module',
 'Buzzer':'Buzzer',
 'Capacitor-10mf':'Capacitor',
 'Capacitor-470mf':'Capacitor',
 'DC-Motor':'DC Motor',
 'Diode':'Diode',
 'ESP32':'ESP32',
 'ESP32-CAM':'ESP32 CAM',
 'FT-232-USB-Serial-Module':'USB Serial Module',
 'Film-Capacitor':'Capacitor',
 'Fuse':'Fuse',
 'Fuse-Base':'Fuse Holder',
 'GSM-Module':'GSM Module',
 'Gas-Sensor':'Gaz',
 'Heat-Sink':'Heat Sink',
 'High-Voltage-Ceramic-Capacitor':'Capacitor',
 'Humidity-Sensor':'Humidity Sensor',
 'IC-Base-14-Pin':'IC Socket',
 'IC-Base-28-Pin':'IC Socket',
 'IC-Chip':'IC',
 'IGBT':'IGBT',
 'IR-Sensor':'IR Sensor',
 'Inductor':'Inductor',
 'Keypad':'Keypad',
 'LCD-Display':'LCD',
 'LDR-Sensor':'LDR Sensor',
 'LED-Light':'LED',
 'Low-Voltage-Ceramic-Capacitor':'Capacitor',
 'MLC-Capacitor':'Capacitor',
 'MOSFET':'MOSFET',
 'Motion-Sensor':'PIR',
 'Motor-Driver':'H Bridge',
 'NTC-Thermistor':'Thermistor',
 'OLED-Display':'LCD OLED',
 'Pin-Header':'Pin Header',
 'Push-Switch':'Button',
 'RFID-Scanner':'RFID',
 'Raindrops-Module':'Rain Sensor',
 'Relay-Module':'Relai',
 'Resistor':'Resistor',
 'Rocker-Switch':'Switch',
 'Servo-Motor':'Servo',
 'Soil-Moisture-Sensor':'Soil Moisture',
 'Sonar-Sensor':'Ultrasonic',
 'TCRT5000':'IR Sensor',
 'Tact-Switch':'Button',
 'Taper-Potentiometer':'Potentiometer',
 'Trimmer-Potentiometer':'Potentiometer',
 'Water-Sensor':'Water Sensor',
 'Zener-Diode':'Diode',
}
INSTRUMENT_ALIASES={
 'breadboard':'Breadboard',
 'component box':'Component Storage Box',
 'digital oscilloscope':'Oscilloscope',
 'function generator':'Function Generator',
 'multimeter':'Multimeter',
 'power strip':'Power Strip',
 'trainer kit':'Electronics Trainer Kit',
}
def resolve(source,original_name):
    if source in ('eesob_original','electronics_original'):
        key=norm(original_name)
        assert key in existing,f'Original class absent from Round 2: {source} / {original_name}'
        return existing[key]
    entries=EXTRA_ALIASES if source=='electronic_component_detection' else INSTRUMENT_ALIASES
    assert original_name in entries,f'NEW CLASS needs manual review: {source} / {original_name}'
    proposed=entries[original_name]
    return existing.get(norm(proposed),proposed)
print('Mapping plan:',len(EXTRA_ALIASES),'expanded part classes,',len(INSTRUMENT_ALIASES),'lab-instrument classes')

## 5 — Load the validated Round 2 YOLO merge reader

Reuses your project's original class-ID parsing and polygon-to-box conversion logic instead of guessing source IDs. This step downloads our public notebook, not any private data.

In [ ]:
import json
round2_notebook_url='https://raw.githubusercontent.com/aquinogeraldedmiranda-art/CircuitVision-YOLO/main/notebooks/CircuitVision_Accuracy_Round2.ipynb'
with urlopen(Request(round2_notebook_url,headers={'User-Agent':'CircuitVision-Training'}),timeout=60) as stream:
    original_nb=json.load(stream)
reader=''.join(original_nb['cells'][7]['source'])
assert all(n in reader for n in ('def read_rows(', 'def source_samples(', 'def read_class_names('))
exec(compile(reader,'round2_merge_reader.py','exec'),globals())
source_data={}
for s in SOURCES:
    name=s['name']
    samples,classes=source_samples(Path(roots[name]),name)
    local_mapped=[resolve(name,c) for c in classes]
    source_data[name]=(samples,classes,local_mapped)
    print(name,':',len(classes),'source classes;','train samples:',len(samples['train']),
          'mapped examples:',dict(list(zip(classes,local_mapped))[:6]))
expanded=sorted({canonical for _,_,mapped in source_data.values() for canonical in mapped if canonical not in old_names})
class_names=old_names+expanded
assert len(set(class_names))==len(class_names),'Duplicate canonical names'
assert class_names[:57]==old_names,'Original indices must not change'
print('Round 3 proposed categories:',len(class_names))
print('New categories:',expanded)

## 6 — Merge sampled images and relabel every box

Source-aware selection protects smaller instrument classes; per-source limits avoid exhausting the free Colab session. **Caution:** Augmented near-duplicates may cross provider splits; this script only removes byte-identical duplicates. Audit dataset origin and leakage before publishing scientific evaluation results.

In [ ]:
from collections import defaultdict
# Per SOURCE sampling limits; change these only if GPU time and disk space allow.
SPLIT_LIMITS={
 'eesob_original':{'train':3200,'val':250,'test':200},
 'electronics_original':{'train':3200,'val':250,'test':200},
 'electronic_component_detection':{'train':2400,'val':250,'test':200},
 'lab_instruments':{'train':1000,'val':100,'test':48},
}
MERGED=Path('/content/circuitvision_round3_merged')
if MERGED.exists():shutil.rmtree(MERGED)
ids={n:i for i,n in enumerate(class_names)}
images_c=Counter();boxes_c=Counter();duplicates=0;missing_annotations=0
rng=random.Random(47);all_hashes=set()
# Test and validation are reserved BEFORE selecting training examples.
for split in ('test','val','train'):
 for spec in SOURCES:
    source=spec['name']
    samples,originals,mapped=source_data[source]
    pool=samples[split].copy()
    rng.shuffle(pool)
    cap=SPLIT_LIMITS[source][split]
    if split=='train':
        # Ensure early coverage of as many labels as the source contains.
        by_class=defaultdict(list)
        for photo in pool:
            for cls,_ in read_rows(photo.labels,len(originals)):
                if len(by_class[cls])<15:by_class[cls].append(photo)
        priority=[];seen_paths=set()
        for class_id in sorted(by_class):
            for photo in by_class[class_id]:
                if photo.path not in seen_paths:
                    priority.append(photo);seen_paths.add(photo.path)
        pool=(priority+[p for p in pool if p.path not in seen_paths])[:cap]
    else:
        pool=pool[:cap]
    for sample in pool:
        digest=hashlib.sha256(sample.path.read_bytes()).hexdigest()
        if digest in all_hashes:
            duplicates+=1;continue
        all_hashes.add(digest)
        annotations=read_rows(sample.labels,len(originals))
        if not sample.labels.exists():missing_annotations+=1
        filename=source+'_'+digest[:18]+sample.path.suffix.lower()
        out_image=MERGED/split/'images'/filename
        out_label=MERGED/split/'labels'/(Path(filename).stem+'.txt')
        out_image.parent.mkdir(parents=True,exist_ok=True)
        out_label.parent.mkdir(parents=True,exist_ok=True)
        shutil.copy2(sample.path,out_image)
        with out_label.open('w') as dest:
            for class_id,xywh in annotations:
                new_name=mapped[class_id]
                dest.write(str(ids[new_name])+' '+' '.join(f'{f:.6f}' for f in xywh)+'\n')
                boxes_c[(split,new_name)]+=1
        images_c[(split,source)]+=1
assert sum(v for (sp,_),v in images_c.items() if sp=='train')>3000
assert sum(v for (sp,_),v in images_c.items() if sp=='test')>300
output_config={'path':str(MERGED.resolve()),'train':'train/images','val':'val/images','test':'test/images','names':class_names,'nc':len(class_names)}
(MERGED/'data.yaml').write_text(yaml.safe_dump(output_config,sort_keys=False))
print('Images by source and split:',dict(images_c))
print('Byte-identical duplicates dropped:',duplicates)
print('Images missing labels (background candidates):',missing_annotations)
print('Labels:',len(class_names),'Original:',len(old_names),'New:',len(expanded))
print('New categories without any training boxes:',[x for x in expanded if boxes_c['train',x]==0])
assert not [x for x in expanded if boxes_c['train',x]==0],'Cannot train classes with no examples'
print('Few-shot new classes:',{x:boxes_c['train',x] for x in expanded if boxes_c['train',x]<25})
print('No new classes were silently omitted')

## 7 — Train a new experimental detector

The original 57 class IDs stay stable, but adding class count changes the detector head and may **reduce** accuracy on some old classes. Weights transfer by matching layers; training and independent testing are essential. Do not update the website automatically.

In [ ]:
from ultralytics import YOLO
TRAIN_EPOCHS=35
TRAIN_IMGSZ=640
TRAIN_BATCH=8
EXPERIMENT_DIR=Path('/content/circuitvision_runs/round3_physical_expansion')
model=YOLO(str(MODEL_PT))
model.train(data=str(MERGED/'data.yaml'),epochs=TRAIN_EPOCHS,imgsz=TRAIN_IMGSZ,
            batch=TRAIN_BATCH,device=0,workers=2,patience=12,seed=47,
            project=str(EXPERIMENT_DIR.parent),name=EXPERIMENT_DIR.name,exist_ok=True,
            amp=True,plots=True,close_mosaic=8,
            mosaic=.65,mixup=.05,degrees=8,translate=.1,scale=.30,fliplr=.5)
BEST=EXPERIMENT_DIR/'weights'/'best.pt'
assert BEST.is_file(),'Round 3 did not produce a trained checkpoint'
print('Candidate Round 3 model:',BEST)

## 8 — Evaluate the NEW model on held-out images

Report per-class AP for both old priorities and new lab tools. A new dataset with additional categories is **not the same benchmark** as Round 2's 49.87% test figure. Hold publication/deployment if key original classes regress badly.

In [ ]:
trained=YOLO(str(BEST))
assert [str(trained.names[i]) for i in range(len(class_names))]==class_names,'Trained class order changed'
metrics=trained.val(data=str(MERGED/'data.yaml'),split='test',imgsz=TRAIN_IMGSZ,device=0,plots=False)
mAP50=float(metrics.box.map50);mAP5095=float(metrics.box.map)
per_ap={class_names[int(i)]:float(metrics.box.ap50[k]) for k,i in enumerate(metrics.box.ap_class_index)}
print('Round 3 candidate mAP@50 (NEW mixed holdout):',round(mAP50,4))
print('Round 3 candidate mAP@50–95:',round(mAP5095,4))
for category in ['Resistor','LED','LDR','Power Module','Diode','IC','Breadboard',
                 'Oscilloscope','Multimeter','Function Generator','ESP32','MOSFET']:
    print(f'{category:22} AP@50:', round(per_ap[category],3) if category in per_ap else 'no evaluable test examples')
print('NOTE: These values are NOT directly comparable with Round 2 mAP 0.4987')
assert mAP50>0,'Empty or unusable benchmark; inspect test annotations'

## 9 — Export candidate weights and class names, but do NOT deploy automatically

The current production backend pins Round 2 by SHA256. We must separately verify any Round 3 ZIP, update the installer, test model accuracy and inference speed, and retain a safe fallback to Round 2.

In [ ]:
import onnxruntime as ort
from pathlib import Path
import json,shutil,zipfile,hashlib
onnx_file=Path(trained.export(format='onnx',imgsz=TRAIN_IMGSZ,opset=17,dynamic=False,nms=False,simplify=True))
session=ort.InferenceSession(str(onnx_file),providers=['CPUExecutionProvider'])
output_shape=session.get_outputs()[0].shape
assert (4+len(class_names)) in output_shape,'ONNX output shape missing expected 4 + new class count'
assert list(session.get_inputs()[0].shape)==[1,3,640,640]
PACK=Path('/content/CircuitVision_Round3_Candidate')
PACK.mkdir(exist_ok=True)
shutil.copy2(BEST,PACK/'best.pt')
shutil.copy2(onnx_file,PACK/'best.onnx')
(PACK/'classes.json').write_text(json.dumps({str(i):n for i,n in enumerate(class_names)},indent=2))
info={'project':'CircuitVision','round':'3 candidate','old_classes':old_names,'new_classes':expanded,
      'all_classes':class_names,'source_limits':SPLIT_LIMITS,'image_counts':{str(k):v for k,v in images_c.items()},
      'duplicate_images_skipped':duplicates,'test_map50':mAP50,'test_map50_95':mAP5095,
      'per_class_ap50':per_ap,'comparison_warning':'Round 3 mixed test set differs from Round 2 test benchmark. Not directly comparable.'}
(PACK/'TRAINING_REPORT.json').write_text(json.dumps(info,indent=2))
(PACK/'DATASET_SOURCES.txt').write_text('\n'.join(
 'https://universe.roboflow.com/'+x['workspace']+'/'+x['project']+'/dataset/'+str(x['version']) for x in SOURCES
)+'\nResistor-band datasets are NOT included in this physical detector.\nVerify licenses before redistribution.\n')
zpath=Path('/content/CircuitVision_Round3_Candidate.zip')
with zipfile.ZipFile(zpath,'w',zipfile.ZIP_DEFLATED) as z:
 for file in PACK.iterdir():z.write(file,arcname=file.name)
print('EXPERIMENTAL package:',zpath)
print('New number of classes:',len(class_names))
print('SHA256:',hashlib.sha256(zpath.read_bytes()).hexdigest())
print('Do not upload this to production until the candidate has been reviewed')

## 10 — Download the candidate and report

On your phone, the ZIP appears in Downloads if Chrome allows the transfer. **Upload the ZIP back in ChatGPT** before any deployment. The original Round 2 website stays untouched.

In [ ]:
from google.colab import files
files.download('/content/CircuitVision_Round3_Candidate.zip')

## Remaining scope and next separate model

Physical 74HC00/74HC08/74HC32/74HC86 ICs are mainly indistinguishable without reading their chip markings; a public candidate dataset containing **HD74LS04** exists (https://universe.roboflow.com/electronic-ic/electronic-component-detection-3ez8s), but it is not sufficient for all logic IC types and was not part of your six-source audit.

A future **resistor band-value reader** should be trained using only band-level annotations and validated against verified resistor values, ideally with close-up photos. The main 57+ physical-component detector and band reader serve different purposes.

Current live site: https://circuitvision-yolo-lab.onrender.com (safe Round 2 until experimental Round 3 passes review).